In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd 
booknow_booking=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv")
booknow_theaters=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/booknow_theaters/booknow_theaters.csv")
booknow_visits= pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/booknow_visits/booknow_visits.csv")
cinePOS_booking=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/cinePOS_booking/cinePOS_booking.csv")
cinePOS_theaters=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/cinePOS_theaters/cinePOS_theaters.csv")
date_info=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/date_info/date_info.csv")
relation=pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/movie_theater_id_relation/movie_theater_id_relation.csv")
sample_submission= pd.read_csv("/kaggle/input/competitions/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv")

## DateTime Parsing

Date columns are loaded as strings by default.
We convert them to datetime and extract only the date part.

- `booknow_visits` → convert `show_date`
- `booknow_booking` → extract date from `show_datetime` and `booking_datetime`
- `cinePOS_booking` → extract date from `show_datetime` and `booking_datetime`
- `date_info` → convert `show_date`

`format='mixed'` is used because dates in the dataset are not in a consistent format.
`dayfirst=True` because dates are in DD/MM/YYYY format.
`.dt.date` extracts only the date part, dropping the time.

In [ ]:
import pandas as pd
import numpy as np
booknow_visits['show_date'] = pd.to_datetime(
    booknow_visits['show_date'], format='mixed', dayfirst=True
).dt.date
booknow_booking['show_date']= pd.to_datetime(
    booknow_booking['show_datetime'],format='mixed',dayfirst=True).dt.date
booknow_booking['booking_date']= pd.to_datetime(
    booknow_booking['booking_datetime'],format='mixed',dayfirst=True).dt.date
cinePOS_booking['show_date']=pd.to_datetime(
    cinePOS_booking['show_datetime'], format='mixed',dayfirst=True).dt.date
cinePOS_booking['booking_date']=pd.to_datetime(
    cinePOS_booking['booking_datetime'], format='mixed',dayfirst=True).dt.date

date_info['show_date'] = pd.to_datetime(
    date_info['show_date'], format='mixed', dayfirst=True
).dt.date

## Aggregation

`booknow_booking` and `cinePOS_booking` are transaction level tables.
Each row is one booking transaction, so one theater can have
multiple rows for the same day.

If we merge directly, we get duplicate rows in our final dataframe.

To fix this, we group by `book_theater_id` + `show_date` and aggregate:
- `total_tickets_booked` → sum of all tickets booked that day
- `num_transactions` → count of booking transactions that day

Same for `cinePOS_booking`:
- `total_tickets_sold` → sum of all tickets sold that day
- `cine_num_transactions` → count of POS transactions that day

Now each theater has exactly one row per day, safe to merge.

In [ ]:
book_agg = booknow_booking.groupby(
    ['book_theater_id', 'show_date']).agg(
    total_tickets_booked = ('tickets_booked', 'sum'),
    num_transactions     = ('tickets_booked', 'count')).reset_index()

cine_agg = cinePOS_booking.groupby(
    ['cine_theater_id', 'show_date']).agg(
    total_tickets_sold    = ('tickets_sold', 'sum'),
    cine_num_transactions = ('tickets_sold', 'count')).reset_index()

## Merging

Base table is `booknow_visits` because it has our target `audience_count`. We use .copy() on this table to avoid modifying the original dataframe while merging.

We left join everything onto it because:
- not every theater uses both booking systems
- left join keeps all rows from base table
- missing data becomes NaN, we handle it in cleaning step

Join order:
1. `book_agg` → on `book_theater_id` + `show_date` (daily booking summary)
2. `booknow_theaters` → on `book_theater_id` (theater metadata)
3. `date_info` → on `show_date` (day of week info)
4. `movie_theater_id_relation` → on `book_theater_id` (to get `cine_theater_id`)
5. `cine_agg` → on `cine_theater_id` + `show_date` (daily POS summary)
6. `cinePOS_theaters` → on `cine_theater_id` (POS theater metadata)

In [ ]:
df=booknow_visits.copy()
df=df.merge(book_agg, on=['book_theater_id','show_date'], how='left')
df=df.merge(booknow_theaters, on='book_theater_id', how='left')
df=df.merge(date_info,on='show_date',how='left')
df=df.merge(relation, on='book_theater_id', how='left')
df=df.merge(cine_agg,on=['cine_theater_id','show_date'],how='left')
df=df.merge(cinePOS_theaters,on='cine_theater_id',how='left')

## Basic Checks

After merging, we check the shape, first few rows and column dtypes
to understand the structure of the combined dataframe.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

## Duplicate Handling

First check for full-row duplicates across all columns.
Then check for duplicate `(book_theater_id, show_date)` combinations
since each theater should have exactly one row per day.

In [ ]:
df.duplicated().sum()

In [ ]:
dupes = df[df.duplicated(keep=False)]
print(dupes)

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Duplicates remaining:", df.duplicated().sum())

In [ ]:
print("theater+date duplicates:",
      df.duplicated(
        subset=['book_theater_id','show_date']).sum())

In [ ]:
dupes = df[df.duplicated(
    subset=['book_theater_id','show_date'], keep=False)]

print(dupes[['book_theater_id','show_date',
             'audience_count']].sort_values(
             ['book_theater_id','show_date']).head(20))

In [ ]:
print(dupes['show_date'].value_counts())

### Fix at Source

All 165 duplicates are on `2023-02-28` in `booknow_visits`.
The audience count for that date was split into two rows per theater.
We combine them using groupby sum so each theater has
one correct total count for that day.
We then verify that `2023-02-28` now has one row per theater
with the combined audience count.

In [ ]:
booknow_visits = booknow_visits.groupby(
    ['book_theater_id', 'show_date'], as_index=False
)['audience_count'].sum()

In [ ]:
print(booknow_visits[booknow_visits['show_date'] == '2023-02-28'].head(10))